# Import packages

In [1]:
from bs4 import BeautifulSoup as bs     # parsing
from urllib.request import urlopen      # scrapping
import pandas as pd                     # exporting
from tenacity import retry, stop_after_attempt
import http.client

In [2]:
from io import StringIO
from pdfminer.converter import TextConverter
from pdfminer.layout import LAParams
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.pdfpage import PDFPage
import os
import logging
from tqdm import tqdm
import csv
import re

In [3]:
address_1 = re.compile('(?:ONE|\d{1,4})?[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)
address_2 = re.compile('(?:ONE|\d{1,4})[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)

city = re.compile('[\w\s]{1,20}[,] \w{1,}?[\d\W]{1,11} [\w\s]{1,3}')
date = re.compile('(?:(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?\s+(?:of\s+)?(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)|(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)\s+(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?)(?:\,)?\s*(?:\d{4})?|[0-3]?\d[-\./][0-3]?\d[-\./]\d{2,4}', re.IGNORECASE)
number = re.compile('\d{1,20}')
title = ['PRESIDENT', 'VICE PRESIDENT', 'VICE PRESIDENT, TAX', 'TREASURER', 'ASSISTANT TREASURER', 'ASST. TREASURER', 'ASSISTANT CLERK', 'CLERK',
         'SECRETARY', 'ASSISTANT SECRETARY', 'CORPORATE SECRETARY', 'GENERAL SECRETARY', 'DIRECTOR', 'SECRETARY & DIRECTOR', 'MEDICAL DIRECTOR',
         'ASST TREAS/SECY', 'EDUCATION DIRECTOR', 'CO-WARDEN', 'VICE PRESIDENT FOOTBALL', 'VICE PRESIDENT CHEER', 'REGISTRAR', 'PUBLIC RELATIONS',
         'SENIOR PASTOR', 'ASSISTANT PASTOR', 'CHURCH LEADER', 'COLLECTOR', 'MODERATOR', 'CHAIR', 'GENERAL MEMBER', 'CEO', 'CFO',
         'GENERAL EVANGELIST', 'EXECUTIVE MINISTER', 'CORP CONTROLLER', 'CORPORATE CONTROLLER', 'OFFICER', 'CHAIRMAN & CEO', 'MANAGER',
         'PRESIDENT & CHIEF OPERATING', 'OPERATIONS', 'EDUCATION', 'CHAIRMAN', 'ICE DIRECTOR', 'CHIEF FINANCIAL OFFICER', 'DIRECTOR OF FINANCE',
         'CHIEF EXECUTIVE OFFICER', 'CHIEF OPERATING OFFICER', 'CHIEF COMPLIANCE OFFICER', 'IMMEDIATE PAST CHAIR', 'CHAIR-ELECT', 'PARTNER',
         'DIRECTOR-CHAIRMAN', 'CFO-TREASURER', 'EVP-COO', 'SENIOR MINISTER', 'MISSIONARY', 'ARCHIVIST', 'SIGNATORY', 'REAL PROPERTY']

# Util functions

In [4]:
def process_pdf(rsrcmgr, device, fp, pagenos=None, maxpages=0, password='', caching=True, check_extractable=True):
    interpreter = PDFPageInterpreter(rsrcmgr, device)
    for page in PDFPage.get_pages(fp, pagenos, maxpages=maxpages, password=password,
                                  caching=caching, check_extractable=check_extractable):
        interpreter.process_page(page)
    return

In [5]:
def read_pdf(pdf):
    rsrcmgr = PDFResourceManager()
    retstr = StringIO()
    laparams = LAParams()
    device = TextConverter(rsrcmgr, retstr, laparams=laparams)
    process_pdf(rsrcmgr, device, pdf)
    device.close()
    content = retstr.getvalue()
    retstr.close()
    lines = str(content).split("\n")
    return lines

In [72]:
def handle(rst, phase):
    '''
    param rst: str, the object representing the info extracted from the pdf, usually the result from function read_pdf
    param phase: int, the phase protocol, to signal which regex compiler should be in use
    '''
#     print(rst)
    idnum = ''
    exactName = ''
    jurisdiction = ''
    registeredName = ''
    registeredAddress = ''
    principalAddress = ''
    directors_info = []
    business = ''
    address = None
    directorsTitle = []
    directorsName = []
    directorsStreet = []
    directorsCity = []

    INFO = []

    if phase == 1:
        address = address_1
    elif phase == 2:
        address = address_2
    
    for i in range(len(rst)):
        if 'Identification Number' in rst[i]:
            idnum = rst[i].split(': ')[1].strip()
        
        if 'Exact name' in rst[i] or 'exact name' in rst[i]:
            if exactName == '':
                exactName = rst[i].split(': ')[1].lstrip()
                
            count = i + 1
            tmp = rst[count]
            # while 'Jurisdiction of Incorporation' not in tmp and 'registered office' not in tmp and \
            #       'principal office' not in tmp:
            #     exactName += tmp.strip()
            #     count += 1
            #     tmp = rst[count]
            #     print(count)
            #     print(tmp)
            
            # exactName = exactName.split('2.')[0].strip()
            
        if 'Jurisdiction of Incorporation' in rst[i]:
            state = ''
            country = ''
            count = i
            tmp = rst[count]
            while 'registered office' not in tmp and 'principal office' not in tmp:
                if 'State' in tmp:
                    state = tmp.split('State: ')[1][0:2]
                if 'Country' in tmp:
                    country = tmp.split('Country: ')[1].strip()
                count += 1
                tmp = rst[count]

            if len(country) != 0:
                jurisdiction = state + ', ' + country
            else:
                jurisdiction = state
            
        if 'corporation registered office' in rst[i] or 'resident agent' in rst[i]:
            registeredStreetList = []
            registeredStreet = ''
            registeredCity = ''
            registeredState = ''
            registeredZip = ''
            registeredCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'principal office' not in tmp and 'directors' not in tmp:
                if 'general partner' in tmp or 'limited partner' in tmp or 'SIGNED'.upper() in tmp.upper():
                   break
                if (address.search(tmp) != None) and (tmp.isupper()):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        registeredStreetList.append(tmp.strip())
                    else:
                        registeredStreetList.append(tmp.split(',')[0].strip())
                elif registeredName == '':
                    if tmp.isupper():
                        registeredName = tmp.strip()
                elif tmp.isupper() and tmp.replace(' ', '').replace('.', '').isalpha():
                    registeredCity = tmp.strip()
                    
                if 'State' in tmp:
                    try:
                        registeredState = tmp.split('State: ')[1].strip()
                    except:
                        registeredState = ''
                if 'Zip' in tmp:
                    try:
                        registeredZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        registeredZip = re.findall('[0-9]+', tmp)[0]
                if 'Country' in tmp:
                    registeredCountry = tmp.split('Country: ')[1].strip()
                    
                count += 1
                tmp = rst[count]
            
            for i in range(len(registeredStreetList)):
                registeredStreet += registeredStreetList[i]
                registeredStreet += ', '
            
            registeredAddress = registeredStreet + registeredCity + ', ' + registeredState + \
                                ' ' + registeredZip + ' ' + registeredCountry
            
        if 'principal office' in rst[i] or 'location' in rst[i].lower() or 'Street address of the office in the Commonwealth'.lower() in rst[i].lower():
            principalStreetList = []
            principalStreet = ''
            principalCity = ''
            principalState = ''
            principalZip = ''
            principalCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'directors' not in tmp:
                if 'name and street address' in tmp or 'limited liability' in tmp:
                   break
                if tmp.isupper() and '3.' not in tmp:
                    if principalStreet == '':
                        if ',' in tmp and tmp.strip().split(',')[1] != '':
                            principalStreet = tmp.strip()
                        else:
                            principalStreet = tmp.split(',')[0].strip()
                if 'CITY' in tmp.upper():
                    principalCity = rst[count+3]
                
                if 'State' in tmp:
                    try:
                        principalState = tmp.split(': ')[1].strip()
                    except:
                        principalState = ''
                if 'Zip' in tmp:
                    try:
                        principalZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        principalZip = ''
                if 'Country' in tmp:
                    principalCountry = tmp.split(': ')[1].strip()
                
                count += 1
                # print(count)
                # print('City', principalCity)
                # print('Street', principalStreet)
                # print('State', principalState)
                # print('Zip', principalZip)
                # print('Country', principalCountry)
                tmp = rst[count]
            
            # for i in range(len(principalStreetList)):
            #     principalStreet += principalStreetList[i]
            #     principalStreet += ', '
            
            principalAddress = principalStreet + ' ' + principalState + \
                               ' ' + principalZip + ' ' + principalCountry
            
        if 'Title' in rst[i]:
            count = i + 1
            tmp = rst[count]
            while 'Briefly describe' not in tmp and 'cemetery corporation' not in tmp and 'undersigned' not in tmp:
                if 'Additional matter'.lower() in tmp.lower() or 'Resident Agent'.lower() in tmp.lower() or 'business address of its principal office' in tmp:
                    break
                # sometimes the title name varies, we can just check if it contains any one in the title list
                if tmp.replace('\xa0', '').strip() in title or \
                bool([t for t in title if (t in tmp.replace('\xa0', '').strip())]): 
                    directorsTitle.append(tmp.strip())
                elif ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                    directorsCity.append(tmp.strip())
                
                # usually the column of name is on the left of the address, so this one step should go first
                elif (tmp.isupper()) and (len(tmp.split()) > 1) and \
                     (re.sub(r'[^\w\s]','',tmp).replace('\xa0', '').replace(' ', '').isalpha()):
                    if address.search(tmp) == None:
                        directorsName.append(tmp.strip())
                    elif ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsName.append(tmp.strip())
                    else:
                        directorsName.append(tmp.split(',')[0].strip())
                        
                elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp) or (tmp.strip() == 'SAME'):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                    
                count += 1
                # print(count)
                # print(tmp)
                tmp = rst[count]
            
            # directorsName = [' '.join(x) for x in zip(directorsName[0::2], directorsName[1::2])] if len(directorsName) >= len(directorsCity) else directorsName
            # directorsCity = [', '.join(x) for x in zip(directorsCity[0::2], directorsCity[1::2])]
            # print(directorsName)
            # print(directorsCity)
            # print(directorsTitle)
            if len(directorsCity) == 0:
                while 'Check' not in tmp and 'Additional matters' not in tmp:
                    if ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                        directorsCity.append(tmp.strip())
                    elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp):
                        directorsStreet.append(tmp.split(',')[0].strip())
                        
                    # print(count)
                    # print(tmp)
                    count += 1
                    tmp = rst[count]
                
            
            directors_info = [[] for _ in range(len(directorsTitle))]

            if len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet) and \
                len(directorsStreet) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                             directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                             directorsStreet[i])
            elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + directorsCity[i])
            elif len(directorsStreet) == len(directorsName) and len(directorsName) == len(directorsCity):
                for i in range(len(directorsStreet)):
                    directors_info[i].append(directorsName[i] + ', ' + \
                                             directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsName):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i])
            elif len(directorsTitle) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsStreet):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsStreet[i])
            elif len(directorsCity) == len(directorsName):
                for i in range(len(directorsName)):
                    directors_info[i].append(directorsName[i] + ', ' + directorsCity[i])
            elif len(directorsCity) == len(directorsStreet):
                for i in range(len(directorsCity)):
                    directors_info[i].append(directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsName) == len(directorsStreet):
                for i in range(len(directorsName)):
                    directors_info[i].append(directorsName[i] + ', ' + directorsStreet[i])
            INFO.extend(directors_info)
                
        if 'The general character' in rst[i] or 'Briefly describe' in rst[i]:
            count = i + 1
            tmp = rst[count]
            while 'cid' in tmp or tmp == ' ' or tmp == '' or 'service, the service to be rendered:' in tmp or \
                address.search(tmp) != None or city.search(tmp) != None or tmp.isupper() == False:
                count += 1
                tmp = rst[count]
                break
            business = tmp.strip()
            
    try:
        id = format(int(idnum), '09d') if len(idnum) <= 9  and len(idnum) else None
    except:
        id = idnum
                    
    rec = {

        'ID Number': id, # fixed length = 9

        'Business Name': exactName,

        'Jurisdiction': jurisdiction,
        
        'Registered Office Name': registeredName,
        
        'Registered Office Address': registeredAddress,
        
        'Principal Office Address': principalAddress,
        
        'Directors': directors_info,
        
        'Business': business
        
        }
    
    return rec

In [73]:
path = '.\PDF_data\PDF4'
pdf = path + '/' + '593773818.pdf'
with open(pdf, 'rb') as f:
    rst = read_pdf(f)
    print(rst)
    info = handle(rst, 1)
    print()
    print(info)

['MA SOC   Filing Number: 202059319890     Date: 2/4/2020 1:28:00 PM', '', 'The Commonwealth of Massachusetts ', '', 'William Francis Galvin ', '', 'Secretary of the Commonwealth, Corporations Division ', '', 'One Ashburton Place, 17th floor ', '', 'Boston, MA 02108-1512 ', 'Telephone: (617) 727-9640 ', '', 'Annual Report\xa0  \xa0', '(General Laws, Chapter 108) ', '', '\xa0   ', '', 'Filing for the last day of February, 2020     ', '', '1. Identification number of the partnership:  593773818 (number will be assigned) ', '', '2. The exact name of the limited liability partnership is:  FEINBERG HANSON LLP  ', '', 'Minimum Fee: $500.00 ', '', '\xa0 ', '', '\xa0  ', '', '3. The street address of its principal office in the Commonwealth: ', 'No. and Street:\xa0  ', 'City or Town: ', '', '855 BOYLSTON ST.  ', 'BOSTON ', '', 'State: MA ', '', '\xa0Zip:   02116 ', '', 'Country: USA ', '', '3. The general character of business, and if the limited liability partnership is organized to render ',

In [74]:
path = '.\PDF_data\PDF'
dirs = os.listdir(path)
# business = {}
results = [] 
for file in tqdm(dirs):
    pdf = path + '/' + file
    with open(pdf, 'rb') as f:
        try:
            rst = read_pdf(f)
        except:
            continue
        try:
            info = handle(rst, 1)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = format(int(file[:-4]), '09d')
            results.append(info)
        except IndexError:
            # print('list index out of range')
            continue
        except ValueError:
            info = handle(rst, 1)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = file[:-4]
            results.append(info)
            continue

100%|████████████████████████████████████████████████████████████████████████████████| 230/230 [02:14<00:00,  1.71it/s]


In [75]:
resultsDF = pd.DataFrame(results)
resultsDF.to_csv('~/Desktop/sec_results.csv')

In [76]:
import shutil

for file in tqdm(resultsDF['ID Number']):
    pdf = path + '/' + file + '.pdf'
    if os.path.exists(pdf):
        shutil.move(pdf, './../Desktop/pdf1-extracted' + '/' + file + '.pdf')

100%|██████████████████████████████████████████████████████████████████████████████| 113/113 [00:00<00:00, 1215.74it/s]


In [77]:
result = pd.read_csv('~/Desktop/results/sec_results1.csv')
result = pd.concat([result, resultsDF], ignore_index=0)
result.to_csv('~/Desktop/sec_results1.csv')

In [78]:
df = pd.read_csv('~/Desktop/sec_results1.csv', index_col=0)
df = df[[col for col in df.columns if col not in ['Unnamed: 0.1', 'Unnamed: 0.1.1', 'Unnamed: 0.1.1.1']]]
df.reset_index(drop=True, inplace=True)
df.to_csv('~/Desktop/sec_results1.csv', index=False)

In [56]:
df = pd.read_csv('~/Desktop/results/sec_results1.csv')

,ID Number,Business Name,Jurisdiction,Registered Office Name,Registered Office Address,Principal Office Address,Directors,Business
0,623657,"IMPAQ/IT, INC.",NaN,NaN,NaN,"37 HATCH ST., MARSHFIELD, MARK R. EPSTEIN, 37 ...","[['VICE PRESIDENT, LEO E SON, 15 ROLAND RD., W...",NaN
1,638932,"A COMMUNITY ONLINE, INC.",NaN,NaN,NaN,"89 SO. OXFORD RD., MILLBURY, PWP,","[['TREASURER, HARLEY H. GOODWIN, 89 SO. OXFORD...",NaN
2,647397,2. The corporation is organized under the laws...,NaN,NaN,NaN,"164 RIVER ROAD, HANOVER, PETER D. MOORE, 164 R...","[['PRESIDENT, CHERYL D. MOORE, , '], ['TREASUR...",NaN
3,655602,"INTERWEB CONCEPTS, INC.",NaN,NaN,NaN,"142 DANA AVE., WORCESTER, MICHAEL J. MORRISSEY...","[['TREASURER, ROBERT P. COONEY, 142 DANA AVE.,...",NaN
4,671379,"ALL RESPONSE CENTER, INC.",NaN,NaN,NaN,"479 MAIN ST., ATHOL, CNP,","[['PRESIDENT, J. DAVID YOUNG, 666 OLD WINCHEST...",NaN
...,...,...,...,...,...,...,...,...
31134,870748857,"IMPERIUM ENTERPRISES, LLC",NaN,NaN,NaN,6 CUDWORTH ST. MA 02155 USA,"[['MANAGER, 6 CUDWORTH ST.'], ['SOC SIGNATORY,...",ACQUISITION AND DEVELOPMENT OF REAL ESTATE.
31135,900133097,"ALDEN DEVELOPMENT ASSOCIATES, LLC",NaN,NaN,NaN,236 WESTERLY CIRCLE MA 01056 USA,"[['MANAGER, 236 WESTERLY CIRCLE'], ['SOC SIGNA...",THE GENERAL CHARACTER OF THE BUSINESS OF THE L...
31136,L00733958,"ABRY CAPITAL PARTNERS, LIMITED PARTNERSHIP",NaN,NaN,"CORPORATION SERVICE COMPANY, 84 STATE STREET, ...",888 BOYLAN STREET SUITE 1600 MA 02199,"[[], [], []]",TO ACT AS THE GENERAL PARTNER OF A LIMITED PAR...
31137,L00733959,"ABRY PARTNERS IV, LIMITED PARTNERSHIP",NaN,NaN,"CORPORATION SERVICE COMPANY, 84 STATE STREET, ...",888 BOYLSTON STREET MA 02199 USA,"[[], [], []]",TO ACT AS A PRIVATE EQUITY LIMITED PARTNERSHIP.


In [57]:
df.drop_duplicates(['ID Number'], ignore_index=True, inplace=True)

In [58]:
df.to_csv('~/Desktop/results/sec_results1.csv', index=False)